In [1]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import BaseCallbackHandler
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter

### Creating a Callback class

In [46]:
class CallCounter(BaseCallbackHandler):
    def __init__(self):
        self.count=0
    
    def on_chat_model_start(self, *args, **kwargs):
        self.count+=1
    
CALLS=CallCounter()

def make_chat_model(model_name="openai/gpt-oss-120b"):
    return ChatGroq(
        model=model_name,
        callbacks=[CALLS]
    )

model=make_chat_model()

In [47]:
def to_document(doc, dept, status, text):
    return Document(
        page_content=text,
        metadata={
            "doc": doc,
            "dept": dept,
            "status": status,
        }
    )

In [48]:
CORPUS = [
    to_document("DOC001", "HR", "Active", "Employee onboarding policy and benefits overview for 2026."),
    to_document("DOC002", "HR", "Draft", "Updated remote work guidelines and home office stipend procedures."),
    to_document("DOC003", "Finance", "Approved", "Q1 2026 financial summary report and budget allocations."),
    to_document("DOC004", "Finance", "Pending", "Vendor invoice processing steps and auditing guidelines."),
    to_document("DOC005", "Engineering", "Active", "System architecture overview and database microservice schema."),
    to_document("DOC006", "Engineering", "Draft", "API security guidelines and OAuth 2.0 authentication flow."),
    to_document("DOC007", "Legal", "Archived", "Standard master services agreement template from 2024."),
    to_document("DOC008", "Legal", "Approved", "Data privacy compliance framework following GDPR and CCPA standards."),
    to_document("DOC009", "Marketing", "Active", "Q3 product launch brand identity and social media campaign assets."),
    to_document("DOC010", "Marketing", "Pending", "Customer feedback survey analysis and market research findings."),
    to_document("DOC011", "Operations", "Active", "Supply chain logistics workflow and warehouse safety guidelines."),
    to_document("DOC012", "Operations", "Draft", "Disaster recovery plan and incident escalation matrix."),
    to_document("DOC013", "Sales", "Approved", "Enterprise sales strategy and competitive analysis matrix."),
    to_document("DOC014", "Support", "Active", "Level 1 customer support troubleshooting manual and FAQ list."),
]

print(len(CORPUS), "documents")

14 documents


# Creating Embedding(L2 Normalized) and Text Splitter(Chunking) Instance

In [5]:
embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

splitter=RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

c:\Users\Shubham Singh\Developer\AIEngineering\pythonAIConcepts\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1789.30it/s]


# Creating a Lambda Function

In [49]:
ACTIVE=lambda d: d.metadata.get("status")=="Active"

def build_all(docs=None):
    global store, chunks
    chunks=splitter.split_documents(docs or CORPUS)
    store=InMemoryVectorStore(embedding=embedding)
    store.add_documents(chunks)

In [50]:
build_all()

print(len(chunks), "chunks")

14 chunks


### Sample Query to get Embedding of Input Text

In [51]:
vector=embedding.embed_query("What's the refund policy")
print(len(vector), vector[:6])

384 [-0.09544149041175842, 0.04163969308137894, 0.0530204251408577, 0.02047557756304741, 0.06370741128921509, 0.053197115659713745]


# Retrieve the Top K Chunks based on Similarity Score

In [52]:
def retrieve(user_query: str, k=3, filter=None):
    kwargs={"filter": filter} if filter else {}
    return store.similarity_search_with_score(user_query, k, **kwargs)

def show(result):
    for doc, score in result:
        print(f"{score:.3f} [{doc.metadata['status']}] {doc.metadata['doc']}: {doc.page_content[:100]}....")

### 1) No additional Metadata for filtering

In [53]:
QUESTION="Find the onboarding policy and benefits overview."
show(retrieve(QUESTION))

0.754 [Active] DOC001: Employee onboarding policy and benefits overview for 2026.....
0.288 [Archived] DOC007: Standard master services agreement template from 2024.....
0.272 [Active] DOC014: Level 1 customer support troubleshooting manual and FAQ list.....


### 2) Providing additional Metadata for filtering

In [66]:
QUESTION="Where can I find the Level 1 customer support troubleshooting manual?"
show(retrieve(QUESTION, k=3, filter=ACTIVE))

0.823 [Active] DOC014: Level 1 customer support troubleshooting manual and FAQ list.....
0.267 [Active] DOC001: Employee onboarding policy and benefits overview for 2026.....
0.265 [Active] DOC011: Supply chain logistics workflow and warehouse safety guidelines.....


In [71]:
SYSTEM_PROMPT="""
You are a helpful assistant for Zorvia Coffee Co. Use only the context below. If the context does not contain the answer, say "I don't know".
Keep your answers concise, short and to the point.
"""

# Prompt template that injects retrieved document contents into {context} and the user's query into {question}.
prompt=ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT+"\n\n"+"Context:\n{context}"),
    ("human", "{question}"),
])

# mini-pipeline that takes a completed prompt dictionary containing context and question,
# sends it to the LLM (model), and parses the output into a plain text string (StrOutputParser()).
answer_chain=prompt | model | StrOutputParser()

# Convert the list of top K retrieved Document objects into a single bulleted string.
def format_docs(docs):
    return "\n".join(f"- [{d.metadata['doc']}] {d.page_content}" for d in docs)

# Builds Langchain Expression Language (LCEL) chain.
def make_chain(retriever):
    return {"context": retriever | format_docs, "question": RunnablePassthrough()} | answer_chain


def naive_rag(user_query, k=3, filter=None):
    retriever=store.as_retriever(
        search_kwargs={"k": k, **({"filter": filter} if filter else {})}
    )

    return make_chain(retriever).invoke(user_query)

In [72]:
naive_rag(QUESTION)

'You can find it in document **[DOC014] – Level\u202f1 Customer Support Troubleshooting Manual and FAQ list**.'